# Pipeline Preditivo de Precificação Imobiliária (King County)

**Opção A – Dataset fornecido:** *kc_house_data.csv*

**Problema de negócio:** Uma imobiliária do condado de King County (EUA) deseja estimar o valor de venda de um imóvel com base em suas características físicas e de localização.

**Variável-alvo:** `price` (valor numérico contínuo em dólares).

**Objetivo do pipeline:** Construir um modelo de regressão capaz de precificar imóveis com acurácia suficiente para apoiar decisões de compra, venda e financiamento, reduzindo o erro médio de estimativa.

---
## Fase 1: Análise Exploratória de Dados (EDA)

Nesta fase, vamos carregar os dados, examinar sua estrutura, calcular estatísticas descritivas e gerar visualizações para compreender os padrões do dataset e orientar as escolhas de modelagem.

### 1.1 Importação das bibliotecas e módulos do projeto

In [ ]:
import sys
from pathlib import Path

# Adiciona a raiz do projeto ao sys.path para importar src/
projeto_raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(projeto_raiz) not in sys.path:
    sys.path.insert(0, str(projeto_raiz))
print(f"Raiz do projeto: {projeto_raiz}")

In [ ]:
# Importação de bibliotecas padrão
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Importação dos módulos do projeto
from src.config import DATASET_FILE, FIGURES_DIR
from src.dataset import carregar_dados, info_dataset, verificar_valores_ausentes
from src.plots import (setup_plot_style, plot_hist_preco,
                       plot_dispersao, plot_matriz_correlacao)

setup_plot_style()
print("Bibliotecas e módulos importados com sucesso.")

### 1.2 Carregamento dos dados

In [ ]:
df = carregar_dados()
print(f"Dataset carregado: {df.shape[0]} linhas e {df.shape[1]} colunas")

In [ ]:
# Primeiras linhas do dataset
df.head()

### 1.3 Estatística Descritiva

In [ ]:
# Dimensões do dataset
print(f"Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas")

In [ ]:
# Tipos primitivos das variáveis
df.dtypes

In [ ]:
# Resumo estatístico descritivo das colunas numéricas
df.describe()

In [ ]:
# Verificação de valores ausentes
ausentes = verificar_valores_ausentes(df)
if ausentes.empty:
    print("Nenhum valor ausente encontrado no dataset.")
else:
    ausentes

### 1.4 Visualização de Dados

#### 1.4.1 Histograma da variável-alvo (price)

In [ ]:
skew_price = plot_hist_preco(df)
print(f"Assimetria (Skewness) de price: {skew_price:.4f}")
plt.show()

#### 1.4.2 Gráficos de dispersão entre variáveis explicativas e a variável-alvo

In [ ]:
# Dispersão: sqft_living vs price
corr1 = plot_dispersao(df, col_x="sqft_living")
print(f"Correlação sqft_living vs price: {corr1:.4f}")
plt.show()

In [ ]:
# Dispersão: grade vs price
corr2 = plot_dispersao(df, col_x="grade")
print(f"Correlação grade vs price: {corr2:.4f}")
plt.show()

In [ ]:
# Dispersão: bathrooms vs price
corr3 = plot_dispersao(df, col_x="bathrooms")
print(f"Correlação bathrooms vs price: {corr3:.4f}")
plt.show()

#### 1.4.3 Mapa de calor da correlação de Pearson (multicolinearidade)

In [ ]:
matriz_correl = plot_matriz_correlacao(df)
plt.show()

In [ ]:
# Exibir pares com correlação absoluta > 0.7 (possível multicolinearidade)
corr_matrix = df.select_dtypes(include="number").corr()
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.7:
            high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j],
                              corr_matrix.iloc[i, j]))

if high_corr:
    print("Pares com alta correlação (|r| > 0.7) — possível multicolinearidade:")
    for par in high_corr:
        print(f"  {par[0]:20s} vs {par[1]:20s}  r = {par[2]:.3f}")
else:
    print("Nenhum par com correlação absoluta > 0.7.")

### 1.5 Análise Textual — Interpretação dos Achados

**Dimensão do dataset:** O conjunto contém aproximadamente 21 mil registros de vendas de imóveis e 21 colunas, oferecendo uma base robusta para treinamento de modelos de regressão.

**Tipos das variáveis:** A maioria das colunas é numérica (int64/float64). As colunas `date` (string) precisará ser convertida para datetime na Fase 2 para extração de atributos temporais. As colunas `waterfront`, `view` e `condition` são codificadas numericamente de forma ordinal, o que as torna diretamente utilizáveis como preditoras.

**Valores ausentes:** Não foram encontrados valores nulos no dataset, o que simplifica a etapa de tratamento de dados.

**Distribuição da variável-alvo (price):** O histograma revela que a distribuição dos preços é ***assimétrica à direita*** (skewness positivo). Isso significa que a maioria dos imóveis tem preços relativamente baixos, enquanto uma cauda longa à direita concentra imóveis de alto valor (possíveis outliers). Essa assimetria pode exigir uma transformação logarítmica na variável-alvo durante a modelagem para melhorar o desempenho do modelo de regressão linear.

**Relações com a variável-alvo:** Os gráficos de dispersão mostram correlações positivas relevantes:
- `sqft_living` (área construída) apresenta forte correlação linear com `price`, sendo a variável preditora mais promissora.
- `grade` (nível de acabamento) também tem boa correlação com o preço, indicando que imóveis com melhor classificação tendem a ser mais caros.
- `bathrooms` (quantidade de banheiros) possui correlação moderada, coerente com a expectativa do mercado.

**Multicolinearidade:** O mapa de calor revela correlações elevadas (|r| > 0.7) entre algumas variáveis explicativas, como `sqft_living` com `sqft_above` e `grade`, e `sqft_living15` com `sqft_living`. Esses pares indicam multicolinearidade que precisará ser tratada na Fase 4, possivelmente com remoção de variáveis redundantes ou análise de VIF (Variance Inflation Factor), para evitar instabilidade nos coeficientes da regressão linear.

**Orientação para as próximas fases:**
- **Fase 2 (Data Prep):** Converter `date` para datetime; tratar outliers de `price`, `sqft_lot` e demais variáveis com distribuições assimétricas.
- **Fase 3 (Feature Engineering):** Criar `idade_imovel`, `foi_reformado` e `preco_por_m2` (esta apenas para análise, sem uso como preditora).
- **Fase 4 (Preparação):** Aplicar encoding em zipcode (agrupamento + One-Hot), remover variáveis com alta multicolinearidade, escalonar com StandardScaler e dividir treino/teste (80/20).

---
*Fim da Fase 1 — Análise Exploratória de Dados*